# De Prompt a Acción: Tu Primer Agente de IA

En este notebook construirás un agente de IA **desde cero**, sin frameworks de alto nivel.
El objetivo es que entiendas exactamente qué ocurre bajo el capó.

## ¿Qué diferencia un LLM de un agente?

| | LLM puro | Agente |
|---|---|---|
| Flujo | `prompt → texto` | `prompt → razonamiento → acción → observación → respuesta` |
| Conocimiento | Estático (hasta cutoff) | Puede buscar información actual |
| Capacidades | Solo lenguaje | Puede ejecutar funciones arbitrarias |

## Lo que construirás

1. ✅ Un cliente configurado (Ollama local **o** API en la nube)
2. ✅ Herramientas (`tools`) definidas como funciones Python puras
3. ✅ El **agentic loop** explícito: la lógica que convierte un LLM en un agente
4. ✅ Un agente con búsqueda web que responde preguntas actuales

---
## ⚙️ Sección 1: Setup

In [1]:
%pip install openai httpx --quiet

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import json
import httpx
import urllib.parse
from openai import OpenAI
from dotenv import load_dotenv
from ddgs import DDGS

load_dotenv()

# ════════════════════════════════════════════════════════════
#  CONFIGURACIÓN — elige tu backend aquí
# ════════════════════════════════════════════════════════════

BACKEND = "nvidia"   # opciones: "ollama" | "anthropic" | "openai" | "groq" | "nvidia" 

CONFIGS = {
    # Ollama corriendo localmente (gratis, sin API key)
    # Instalar: https://ollama.com  →  ollama pull qwen2.5:7b
    "ollama": {
        "base_url": "http://localhost:11434/v1",
        "api_key": "ollama",           # requerido por el cliente, ignorado por Ollama
        "model":   "qwen2.5:7b",       # alternativas: llama3.1, mistral-nemo
    },
    # Anthropic vía endpoint OpenAI-compatible
    # Requiere: ANTHROPIC_API_KEY en entorno
    "anthropic": {
        "base_url": "https://api.anthropic.com/v1",
        "api_key": os.getenv("ANTHROPIC_API_KEY", ""),
        "model":   "claude-3-5-haiku-20241022",
    },
    # OpenAI
    # Requiere: OPENAI_API_KEY en entorno
    "openai": {
        "base_url": None,              # usa el endpoint por defecto de OpenAI
        "api_key": os.getenv("OPENAI_API_KEY", ""),
        "model":   "gpt-4o-mini",
    },
    # Groq (muy rápido, tier gratuito disponible)
    # Requiere: GROQ_API_KEY en entorno  →  https://console.groq.com
    "groq": {
        "base_url": "https://api.groq.com/openai/v1",
        "api_key": os.getenv("GROQ_API_KEY", ""),
        "model":   "llama-3.3-70b-versatile",
    },
    # NVIDIA NIM (catálogo de 50+ modelos, tier gratuito con 1,000 créditos)
    # Requiere: NVIDIA_API_KEY en entorno  →  https://build.nvidia.com
    # El API key empieza con nvapi-
    # Modelos con tool use: meta/llama-3.3-70b-instruct, mistralai/mistral-large-latest
    "nvidia": {
        "base_url": "https://integrate.api.nvidia.com/v1",
        "api_key":  os.getenv("NVIDIA_API_KEY", ""),
        "model":    "meta/llama-3.3-70b-instruct",  # o cualquier otro del catálogo
    },
}

cfg = CONFIGS[BACKEND]

client_kwargs = {"api_key": cfg["api_key"]}
if cfg["base_url"]:
    client_kwargs["base_url"] = cfg["base_url"]

client = OpenAI(**client_kwargs)
MODEL  = cfg["model"]

print(f"✅ Backend: {BACKEND.upper()} | Modelo: {MODEL}")

✅ Backend: NVIDIA | Modelo: meta/llama-3.3-70b-instruct


> **Nota:** usamos el cliente `openai` para todos los backends porque la mayoría
> de proveedores (Ollama, Groq, Anthropic, Together, etc.) exponen una API
> compatible con el formato de OpenAI. Esto hace que el agentic loop sea
> **idéntico** sin importar qué modelo uses.

---
## Sección 2: Definir Herramientas

Una **herramienta** (tool) tiene dos partes:

1. **La función Python** — la implementación real que se ejecuta
2. **El JSON schema** — la descripción que el LLM lee para saber cuándo y cómo llamarla

El modelo **nunca ejecuta** la función directamente. Solo devuelve un `tool_calls`
con el nombre y los argumentos. Nosotros somos quienes la ejecutamos y devolvemos
el resultado. Eso es exactamente lo que hará el agentic loop.

In [7]:
# ── Implementaciones reales ──────────────────────────────────────────────────

def web_search(query: str) -> str:
    """Busca información actualizada en la web."""
    try:
        results = DDGS().text(query, max_results=3)
        if not results:
            return "No se encontraron resultados."
        output = []
        for r in results:
            output.append(f"**{r['title']}**\n{r['body']}")
        return "\n\n".join(output)
    except Exception as e:
        return f"Error al buscar: {e}"


def get_weather(city: str) -> str:
    """Obtiene el clima actual de una ciudad usando wttr.in (sin API key)."""
    try:
        r = httpx.get(f"https://wttr.in/{urllib.parse.quote(city)}?format=3", timeout=8)
        return r.text.strip()
    except Exception as e:
        return f"Error al obtener clima: {e}"


def calculate(expression: str) -> str:
    """Evalúa una expresión matemática simple."""
    try:
        allowed = set("0123456789+-*/.() ")
        if not all(c in allowed for c in expression):
            return "Error: solo se permiten expresiones numéricas básicas."
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        return f"Error al calcular: {e}"


# ── Registro: nombre → función Python ────────────────────────────────────────
# Este dict es el "router" del agentic loop.
TOOL_REGISTRY = {
    "web_search": web_search,
    "get_weather": get_weather,
    "calculate":  calculate,
}

print("✅ Funciones de herramientas definidas.")

✅ Funciones de herramientas definidas.


In [9]:
# ── JSON schemas: lo que el modelo ve ────────────────────────────────────────
# Formato OpenAI tool use — compatible con Ollama, Groq, OpenAI y Anthropic.

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": (
                "Busca información actualizada en la web. "
                "Úsala cuando necesites datos recientes o hechos que podrían "
                "haber cambiado después de tu fecha de corte."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string",
                              "description": "La consulta de búsqueda en lenguaje natural."}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Obtiene el clima actual de una ciudad específica.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string",
                             "description": "Nombre de la ciudad, ej: 'Medellín' o 'London'."}
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evalúa expresiones matemáticas. Úsala para cálculos precisos.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string",
                                   "description": "Expresión matemática, ej: '(15 * 8) / 3 + 7'."}
                },
                "required": ["expression"]
            }
        }
    }
]

print(f"✅ {len(TOOLS)} herramientas: {[t['function']['name'] for t in TOOLS]}")

✅ 3 herramientas: ['web_search', 'get_weather', 'calculate']


---
## 🔄 Sección 3: El Agentic Loop

Este es el núcleo de cualquier agente. El loop hace lo siguiente en cada iteración:

```
┌─────────────────────────────────────────────────────┐
│                   AGENTIC LOOP                      │
│                                                     │
│  messages ──► LLM                                   │
│                │                                    │
│         ┌──────┴──────────────┐                     │
│   finish_reason=stop   finish_reason=tool_calls     │
│         │                      │                    │
│      RETORNAR           ejecutar función            │
│                                │                    │
│                      agregar tool result            │
│                      a messages y repetir           │
└─────────────────────────────────────────────────────┘
```

**Puntos clave:**
- El historial completo se envía en cada llamada (el LLM es stateless)
- Un solo prompt puede provocar múltiples tool calls
- El loop termina cuando `finish_reason == "stop"`

In [10]:
def run_agent(
    user_message: str,
    system: str = None,
    tools: list = None,
    tool_registry: dict = None,
    verbose: bool = True,
    label: str = "Agente",
) -> str:
    """
    Ejecuta el agentic loop para un mensaje de usuario.

    Args:
        user_message:  La pregunta o instrucción del usuario.
        system:        System prompt opcional.
        tools:         Lista de tool schemas (formato OpenAI). Por defecto: TOOLS.
        tool_registry: Dict nombre→función. Por defecto: TOOL_REGISTRY.
        verbose:       Si True, imprime cada paso.
        label:         Etiqueta para los prints (útil en sistemas multi-agente).

    Returns:
        Texto final de respuesta del agente.
    """
    tools         = tools         if tools         is not None else TOOLS
    tool_registry = tool_registry if tool_registry is not None else TOOL_REGISTRY

    if system is None:
        system = (
         "Eres un asistente útil con acceso a herramientas. "
         "Usa las herramientas SOLO cuando necesites información que cambia con el tiempo "
         "(noticias, precios, clima, eventos recientes). "
         "Para conocimiento general o hechos estables, responde directamente sin usar tools. "
         "Responde en el idioma del usuario."
         )

    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": user_message},
    ]

    if verbose:
        print(f"\n{'─'*60}")
        print(f"  [{label}] 👤 {user_message[:120]}")

    for iteration in range(1, 11):  # máx 10 iteraciones

        # ── Llamada al LLM ──────────────────────────────────────────────
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools if tools else None,
            tool_choice="auto" if tools else None,
        )

        choice        = response.choices[0]
        finish_reason = choice.finish_reason

        if verbose:
            print(f"  [{label}] iter={iteration} finish_reason={finish_reason}")

        # ── ¿Terminó? ───────────────────────────────────────────────────
        if finish_reason == "stop":
            text = choice.message.content or ""
            if verbose:
                preview = text[:200] + "..." if len(text) > 200 else text
                print(f"  [{label}] 🤖 {preview}")
            return text

        # ── ¿Hay tool calls? ────────────────────────────────────────────
        if finish_reason == "tool_calls":
            assistant_msg = choice.message

            # 1. Agregar la respuesta del asistente al historial
            messages.append(assistant_msg)

            # 2. Ejecutar cada tool call y agregar el resultado
            for tc in assistant_msg.tool_calls:
                tool_name = tc.function.name
                tool_args = json.loads(tc.function.arguments)

                if verbose:
                    print(f"  [{label}] 🔧 {tool_name}({json.dumps(tool_args, ensure_ascii=False)})")

                fn = tool_registry.get(tool_name)
                result = fn(**tool_args) if fn else f"Error: herramienta '{tool_name}' no encontrada."

                if verbose:
                    preview = result[:150] + "..." if len(result) > 150 else result
                    print(f"  [{label}]    → {preview}")

                # 3. Agregar el resultado al historial
                messages.append({
                    "role":         "tool",
                    "tool_call_id": tc.id,
                    "content":      result,
                })

        else:
            break  # finish_reason inesperado

    return "[Agente detenido: se alcanzó el límite de iteraciones]"


print("✅ Agentic loop definido.")

✅ Agentic loop definido.


---
## 🧪 Sección 4: Prueba el Agente

### 4.1 Pregunta sin herramientas
El modelo responde directo — `finish_reason=stop` en la primera iteración.

In [11]:
run_agent("¿Cuántas lunas tiene Marte?")


────────────────────────────────────────────────────────────
  [Agente] 👤 ¿Cuántas lunas tiene Marte?
  [Agente] iter=1 finish_reason=tool_calls
  [Agente] 🔧 web_search({"query": "¿Cuántas lunas tiene Marte?"})
  [Agente]    → **Satélites de Marte - Wikipedia, la enciclopedia libre**
July 4, 2025 - Cabe resaltar que, a diferencia de la Luna terrestre, estos satélites no disi...
  [Agente] iter=2 finish_reason=stop
  [Agente] 🤖 Marte tiene dos lunas, llamadas Fobos y Deimos.


'Marte tiene dos lunas, llamadas Fobos y Deimos.'

### 4.2 Búsqueda web

In [12]:
run_agent("¿Qué es el Agent Development Kit de Google?")


────────────────────────────────────────────────────────────
  [Agente] 👤 ¿Qué es el Agent Development Kit de Google?


  [Agente] iter=1 finish_reason=tool_calls
  [Agente] 🔧 web_search({"query": "Agent Development Kit de Google"})
  [Agente]    → **Agent Development Kit (ADK) - Agent Development Kit (ADK)**
1 month ago - For enterprises, ADK can connect to models on hosted services, including G...
  [Agente] iter=2 finish_reason=stop
  [Agente] 🤖 El Agent Development Kit (ADK) de Google es un marco de desarrollo de agentes de código abierto que permite a los desarrolladores crear, depurar y desplegar agentes de inteligencia artificial confiabl...


'El Agent Development Kit (ADK) de Google es un marco de desarrollo de agentes de código abierto que permite a los desarrolladores crear, depurar y desplegar agentes de inteligencia artificial confiables a escala empresarial. Está diseñado para ser flexible y fácil de usar, y proporciona una base sólida para construir la próxima generación de aplicaciones de inteligencia artificial.'

### 4.3 Clima

In [13]:
run_agent("¿Cómo está el clima en Medellín ahora?")


────────────────────────────────────────────────────────────
  [Agente] 👤 ¿Cómo está el clima en Medellín ahora?
  [Agente] iter=1 finish_reason=tool_calls
  [Agente] 🔧 get_weather({"city": "Medellín"})
  [Agente]    → medellín: ⛅  +69°F
  [Agente] iter=2 finish_reason=stop
  [Agente] 🤖 El clima en Medellín ahora es de 69°F con un clima nublado.


'El clima en Medellín ahora es de 69°F con un clima nublado.'

### 4.4 Múltiples herramientas encadenadas

In [14]:
run_agent(
    "Busca el precio actual de Bitcoin en USD y calcula cuánto costarían 0.75 BTC."
)


────────────────────────────────────────────────────────────
  [Agente] 👤 Busca el precio actual de Bitcoin en USD y calcula cuánto costarían 0.75 BTC.
  [Agente] iter=1 finish_reason=tool_calls
  [Agente] 🔧 web_search({"query": "precio actual de Bitcoin en USD"})
  [Agente]    → **BTC a USD: intercambiar Bitcoin (BTC) a Dólar estadounidense (USD)**
Intercambiar Bitcoin BTC a United States Dollar USD. BTC a USD: 1 Bitcoin se co...
  [Agente] iter=2 finish_reason=tool_calls
  [Agente] 🔧 calculate({"expression": "0.75 * 76260"})
  [Agente]    → 57195.0
  [Agente] iter=3 finish_reason=stop
  [Agente] 🤖 El precio actual de Bitcoin es de $76,260.00 USD por BTC. Por lo tanto, 0.75 BTC costarían $57,195.00 USD.


'El precio actual de Bitcoin es de $76,260.00 USD por BTC. Por lo tanto, 0.75 BTC costarían $57,195.00 USD.'

---
## 🚀 Sección 5: Tu Turno

Experimenta:
- ¿Qué pasó en las noticias hoy?
- ¿Cuánto es el 18% de 3.450.000?
- ¿Cuál es el clima en Tokyo?
- Una pregunta que requiera búsqueda **y** cálculo

In [ ]:
run_agent("...")

---
## 🔬 Sección 6: Bajo el Capó — Inspección del Historial

Expone el historial completo de mensajes para ver exactamente
qué se envía al modelo en cada vuelta del loop.

In [15]:
def run_agent_inspect(user_message: str) -> list:
    """Igual que run_agent pero retorna el historial completo."""
    messages = [
        {"role": "system", "content": "Eres un asistente útil. Usa las herramientas cuando sea necesario."},
        {"role": "user",   "content": user_message},
    ]
    for _ in range(10):
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=TOOLS, tool_choice="auto"
        )
        choice = response.choices[0]
        if choice.finish_reason == "stop":
            messages.append({"role": "assistant", "content": choice.message.content})
            break
        if choice.finish_reason == "tool_calls":
            messages.append(choice.message)
            for tc in choice.message.tool_calls:
                fn   = TOOL_REGISTRY.get(tc.function.name)
                args = json.loads(tc.function.arguments)
                result = fn(**args) if fn else "herramienta no encontrada"
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
    return messages


history = run_agent_inspect("¿Cuál es el clima en Bogotá?")

print("=" * 70)
print("HISTORIAL COMPLETO DE MENSAJES")
print("=" * 70)
for i, msg in enumerate(history):
    if isinstance(msg, dict):
        role    = msg.get("role", "?").upper()
        content = str(msg.get("content", ""))
        print(f"\n[{i}] {role}: {content[:300]}")
    else:
        role = (msg.role or "?").upper()
        print(f"\n[{i}] {role}:")
        if msg.content:
            print(f"  text: {msg.content[:300]}")
        if getattr(msg, "tool_calls", None):
            for tc in msg.tool_calls:
                print(f"  tool_call: {tc.function.name}({tc.function.arguments[:200]})")

HISTORIAL COMPLETO DE MENSAJES

[0] SYSTEM: Eres un asistente útil. Usa las herramientas cuando sea necesario.

[1] USER: ¿Cuál es el clima en Bogotá?

[2] ASSISTANT:
  tool_call: get_weather({"city": "Bogotá"})

[3] TOOL: bogotá: ⛅  +56°F

[4] ASSISTANT: El clima en Bogotá es de 56°F con cielos parcialmente nublados.


---
## ✅ Resumen

Construiste un agente desde cero. Los conceptos clave:

- **Tools** = funciones Python + JSON schema que las describe al LLM
- **Agentic loop** = `while finish_reason == "tool_calls"`: ejecuta tools, agrega resultados, llama al LLM de nuevo
- **Historial de mensajes** = el estado completo que se envía en cada llamada
- El mismo código funciona con Ollama, OpenAI, Groq o Anthropic — solo cambia `BACKEND` en la celda de configuración

### Próximos pasos

En el siguiente notebook verás cómo componer múltiples agentes:
pipelines secuenciales, ejecución paralela y loops de refinamiento.